In [1]:
# 🛑 [셀 1] 패키지 강제 세팅 및 필수 폴더/모델 다운로드 (가장 먼저 단독으로 실행!)
import os

print("▶️  시스템 패키지 설치 중...")
!apt-get update -y && apt-get install -y mecab libmecab-dev mecab-ipadic-utf8 curl git

print("▶️ GPT-SoVITS 프로그램 폴더 다운로드 중...")
os.chdir('/kaggle/working')
if not os.path.exists("GPT-SoVITS"):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS.git

print("▶️ V3 사전 학습 모델(가중치) 다운로드 중...")
os.makedirs("/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models", exist_ok=True)
os.chdir('/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models')
!wget -q -O s1v3.ckpt https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1v3.ckpt
!wget -q -O s2Gv3.pth https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s2Gv3.pth

print("▶️ 필수 파이썬 패키지 강제 다운그레이드 및 설치 중...")
!pip install -q -U huggingface_hub gradio
!pip install -q mlflow dagshub gTTS soundfile
# 🔥 [핵심 추가 패치] split_lang 등 백서에 기록된 '숨은 연쇄 누락 패키지' 6종 싹 다 추가!
!pip install -q split_lang x_transformers pytorch_lightning cn2an pypinyin jieba_fast
!pip install -q --force-reinstall mecab-python3 python-mecab-ko mecab-ko-dic g2pk2 ko_pron fast_langdetect wordsegment g2p_en konlpy
# 💡 핵심: 최신 패키지들이 깔린 후, 제일 마지막에 문제가 되던 Numpy와 fsspec을 구버전으로 덮어씌웁니다!
!pip install -q --force-reinstall numpy==1.26.4 fsspec==2025.3.0

print("\n✅ 패키지 및 모델 디스크 설치 완벽 성공! 이제 반드시 화면 상단의 둥근 화살표 [Restart]를 눌러 커널을 재시작하세요.")


▶️  시스템 패키지 설치 중...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:13 https://ppa.launchpadcontent

2단계: 커널 재시작 (Restart)


3단계: 새로운 셀을 만들고, 아래의 [셀 2: 순수 실행 전용] 코드를 넣고 실행하세요.


RoBERTa는 Meta(구 페이스북)에서 개발한 자연어 처리(NLP) 언어 모델로, 기존 구글의 BERT 모델을 더 크고 정교하게 최적화하여 성능을 크게 끌어올린 모델입
HuBERT(Hidden-Unit BERT)는 2021년 메타(구 페이스북) AI 연구진이 발표한 BERT의 구조와 학습 방식을 '음성(Speech)' 영역에 적용한 인공지능 모델

In [2]:
# 📥 [긴급 패치] 누락된 텍스트 분석 기반 모델(RoBERTa 등) 전체 다운로드
import os
from huggingface_hub import snapshot_download

print("▶️ 텍스트 분석용 기반 모델(RoBERTa, HuBERT 등) 전체 다운로드 중 (약 1~2분 소요)...")
os.makedirs("/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models", exist_ok=True)

snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir="/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models",
    local_dir_use_symlinks=False
)

print("✅ 모든 기반 모델 다운로드 완벽 성공! 이제 [셀 2: 순수 실행 전용] 코드를 다시 실행해 주세요!")


▶️ 텍스트 분석용 기반 모델(RoBERTa, HuBERT 등) 전체 다운로드 중 (약 1~2분 소요)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

✅ 모든 기반 모델 다운로드 완벽 성공! 이제 [셀 2: 순수 실행 전용] 코드를 다시 실행해 주세요!


In [3]:
# 🚀 [셀 2] 커널 재시작 후 실행하는 순수 파이썬 연산 코드 (설치 명령어 없음!)
import os
import sys
import gc
import torch
import soundfile as sf
import mlflow
import dagshub
import nltk
from gtts import gTTS
from IPython.display import Audio, display

print("▶️ [0단계] NLTK 영어 발음 및 문법 분석 데이터 다운로드 중...")
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('cmudict', quiet=True)

print("▶️ [1단계] 켜져있는 API 서버 강제 종료 및 GPU 메모리 대청소...")
!pkill -f api.py
gc.collect()
if torch.cuda.is_available(): 
    torch.cuda.empty_cache()

os.chdir('/kaggle/working/GPT-SoVITS')
if '/kaggle/working/GPT-SoVITS' not in sys.path:
    sys.path.append('/kaggle/working/GPT-SoVITS')
if '/kaggle/working/GPT-SoVITS/GPT_SoVITS' not in sys.path:
    sys.path.append('/kaggle/working/GPT-SoVITS/GPT_SoVITS')

print("▶️ [1.2단계] 언어 감지 모델용 캐시 폴더 강제 생성 중...")
os.makedirs("/kaggle/working/GPT-SoVITS/GPT_SoVITS/pretrained_models/fast_langdetect", exist_ok=True)

# [패치 1] 한국어 언어코드 강제 주입
import inference_webui
from inference_webui import change_gpt_weights, change_sovits_weights, get_tts_wav, dict_language
dict_language["ko"] = "all_ko"
dict_language["한국어"] = "all_ko"

class DummyData: sampling_rate = 32000
class DummyHPS: data = DummyData()
inference_webui.hps = DummyHPS()

print("▶️ [1.5단계] 이미 생성된 번역기(g2pk2) 객체에 Mecab 뇌 직접 이식 중...")
import text.korean
from mecab import MeCab
import mecab_ko_dic

try:
    dic_path = mecab_ko_dic.dictionary_path
except AttributeError:
    dic_path = "/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dicdir"

text.korean._g2p.mecab = MeCab(dictionary_path=dic_path) 
print("✅ g2pk2 다이렉트 뇌 이식 완벽 성공!")

print("▶️ [1.8단계] 파이썬 캐시 무시! BigVGAN 성대에 강제로 누락된 스위치를 달아줍니다...")
if hasattr(inference_webui, 'bigvgan') and hasattr(inference_webui.bigvgan.BigVGAN, '_from_pretrained'):
    orig_from_pretrained = inference_webui.bigvgan.BigVGAN._from_pretrained.__func__
    @classmethod
    def patched_from_pretrained(cls, *args, **kwargs):
        kwargs.setdefault('proxies', None)
        kwargs.setdefault('resume_download', False)
        return orig_from_pretrained(cls, *args, **kwargs)
    inference_webui.bigvgan.BigVGAN._from_pretrained = patched_from_pretrained
    print("✅ BigVGAN 메모리 수술(백도어) 완벽 성공!")

print("▶️ [2단계] V3 모델 가중치 파일 경로 고정 지정 중...")
REAL_GPT_PATH = "GPT_SoVITS/pretrained_models/s1v3.ckpt"
REAL_SOVITS_PATH = "GPT_SoVITS/pretrained_models/s2Gv3.pth"

print("▶️ [3단계] 5.5초 황금비율 기준 오디오 생성 중...")
medium_prompt = "안녕하세요. 지금부터 인공지능 음성 합성 테스트를 시작하겠습니다."
if not os.path.exists("ref_v6.wav"):
    tts_ref = gTTS(medium_prompt, lang="ko")
    tts_ref.save("ref_temp_v6.mp3")
    import subprocess
    subprocess.run(["ffmpeg", "-i", "ref_temp_v6.mp3", "-ar", "16000", "-ac", "1", "-y", "ref_v6.wav", "-loglevel", "quiet"])

print("▶️ [4단계] 차라리 직접 조립합니다! 메모리에 부품 순차 적재 중...")
gpt_loader = change_gpt_weights(REAL_GPT_PATH)
if hasattr(gpt_loader, '__iter__') and not isinstance(gpt_loader, str):
    for msg in gpt_loader: pass  
        
sovits_loader = change_sovits_weights(REAL_SOVITS_PATH, "ko", "ko") 
if hasattr(sovits_loader, '__iter__') and not isinstance(sovits_loader, str):
    for msg in sovits_loader: pass  

if not hasattr(inference_webui, 'vq_model') or inference_webui.vq_model is None:
    raise RuntimeError("❌ 앗! vq_model 부품이 여전히 없습니다. 로딩 에러 메시지를 확인하세요.")

print("✅ 모든 부품(ssl_model, vq_model, hps) 메모리 적재 완벽 성공!")

print("▶️ [5단계] DagsHub 연동 및 직통 음성 합성...")
dagshub.init(repo_owner='myelin24m', repo_name='Kride.mlflow', mlflow=True)
mlflow.set_experiment("7")

text_to_speak = "안녕하세요! 마침내 지독했던 에러를 극복하고 AI 음성 합성에 성공했습니다. 정말 고생 많으셨습니다!"
print(f"🎙️ 텍스트 변환 직통 연산 중: '{text_to_speak}'")

try:
    generator = get_tts_wav(
        ref_wav_path="ref_v6.wav",
        prompt_text=medium_prompt,
        prompt_language="ko",
        text=text_to_speak,
        text_language="ko"
    )
    
    sr, audio_data = next(generator)
    
    output_file = "final_output.wav"
    sf.write(output_file, audio_data, sr)
    print(f"✅ 오디오 파일 생성 완료! ({output_file})")

    with mlflow.start_run(run_name="Kaggle_Direct_Assembly_Success"):
        mlflow.log_param("text", text_to_speak)
        mlflow.log_artifact(output_file)
    print("✅ DagsHub(MLflow) 오디오 기록 완벽 성공!")

    print("\n🎧 아래 재생 버튼을 눌러 AI의 목소리를 확인해 보세요!")
    display(Audio(output_file))

except Exception as e:
    print("\n❌ 🚨 직통 연산 중 에러 발생! 상세 원인:")
    import traceback
    traceback.print_exc()

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# 🛑 [핵심 패치] GPT-SoVITS 내부 연산에 필요한 ffmpeg 패키지 추가!
!pip install -q ffmpeg-python

import sys
import soundfile as sf
import os
import time
import mlflow
import getpass
import numpy as np
from IPython.display import Audio, display

print("▶️ [0단계] DagsHub 연동 설정 중...")
# 🔥 브라우저 팝업을 띄우지 않고 토큰을 직접 입력받습니다!
os.environ['MLFLOW_TRACKING_USERNAME'] = "myelin24m"
# 실행 시 비밀번호 입력창이 뜨면 DagsHub Access Token을 붙여넣고 Enter를 누르세요!
os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass.getpass('🔑 DagsHub Access Token을 붙여넣고 Enter를 누르세요: ')

tracking_uri = "https://dagshub.com/myelin24m/Kride.mlflow"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("7")

# 기존 실행 중인 꼬인 MLflow 세션이 있다면 안전하게 종료
if mlflow.active_run():
    mlflow.end_run()

print("▶️ [1단계] GPT-SoVITS 경로 인식 중...")
os.chdir('/kaggle/working/GPT-SoVITS')
GPT_SOVITS_REPO_ROOT = '/kaggle/working/GPT-SoVITS'
if GPT_SOVITS_REPO_ROOT not in sys.path:
    sys.path.append(GPT_SOVITS_REPO_ROOT)

GPT_SOVITS_PACKAGE_INNER_DIR = os.path.join(GPT_SOVITS_REPO_ROOT, 'GPT_SoVITS')
if GPT_SOVITS_PACKAGE_INNER_DIR not in sys.path:
    sys.path.append(GPT_SOVITS_PACKAGE_INNER_DIR)

print("▶️ [2단계] 허깅페이스 대문에 전역 백도어 트랩 설치 중...")
import huggingface_hub.hub_mixin
if not hasattr(huggingface_hub.hub_mixin.ModelHubMixin, '_is_global_patched'):
    orig_from_pretrained = huggingface_hub.hub_mixin.ModelHubMixin.from_pretrained.__func__
    
    @classmethod
    def patched_from_pretrained(cls, *args, **kwargs):
        if cls.__name__ == "BigVGAN" and not getattr(cls, "_is_local_patched", False):
            orig_cls_from_pretrained = cls._from_pretrained.__func__
            @classmethod
            def safe_from_pretrained(subcls, *a, **kw):
                kw.setdefault('proxies', None)
                kw.setdefault('resume_download', False)
                return orig_cls_from_pretrained(subcls, *a, **kw)
            cls._from_pretrained = safe_from_pretrained
            cls._is_local_patched = True
        return orig_from_pretrained(cls, *args, **kwargs)
        
    huggingface_hub.hub_mixin.ModelHubMixin.from_pretrained = patched_from_pretrained
    huggingface_hub.hub_mixin.ModelHubMixin._is_global_patched = True
    print("✅ huggingface_hub 전역 백도어 수술 완벽 성공!")
else:
    print("✅ 이미 백도어가 설치되어 있습니다!")

# 실제 추론 모듈 로드
from inference_webui import get_tts_wav

# 테스트할 내레이션과 기존에 생성해둔 레퍼런스 오디오 경로
test_text = "안녕하세요! 마침내 지독했던 에러를 극복하고 AI 음성 합성에 성공했습니다. 정말 고생 많으셨습니다!"
medium_prompt = "안녕하세요. 지금부터 인공지능 음성 합성 테스트를 시작하겠습니다."
dummy_ref_audio_path = "ref_v6.wav" 

print("\n▶️ [3단계] GPT-SoVITS 추론 및 DagsHub(MLflow) 성능 기록 시작...")
with mlflow.start_run(run_name="Test_GPT_SoVITS_Final"):
    mlflow.log_param("model_name", "GPT-SoVITS")
    mlflow.log_param("text_length", len(test_text))
    start_time = time.time()
    output_file_path = "gpt_sovits_final_output.wav"
    
    try:
        print(f"🎤 변환할 텍스트: '{test_text}'")
        generator = get_tts_wav(
            ref_wav_path=dummy_ref_audio_path, 
            prompt_text=medium_prompt, 
            prompt_language="ko",
            text=test_text,
            text_language="ko"
        )
        
        sr, audio_data = next(generator)
        sf.write(output_file_path, audio_data, sr)
        
        execution_time = time.time() - start_time
        mlflow.log_metric("generation_time_sec", execution_time)
        mlflow.log_artifact(output_file_path, "audio_results")
        
        print(f"\n✅ 대성공! 오디오 파일이 생성되었습니다: {output_file_path}")
        print(f"⏱️ 소요 시간: {execution_time:.2f}초")
        print("👉 DagsHub(MLflow) 대시보드에 소요 시간과 오디오 결과물이 완벽하게 기록되었습니다!")
        
        print("\n🎧 아래 재생 버튼을 눌러 피땀 흘려 만든 AI의 목소리를 확인해 보세요!")
        display(Audio(output_file_path))
        
    except Exception as e:
        print(f"\n❌ 🚨 합성 중 오류가 발생했습니다: {str(e)}")
        import traceback
        traceback.print_exc()